# **[Comparative Analysis] Ablation Study Synthesis (yunnan-garden-pavilion-P71)**

##### **The Objective: Comparative Evaluation**
The objective of this analysis is to synthesize the architectural impact of the Phase 2 ablation studies against the Phase 1 geometric baseline. By mapping the performance of each feature matching architecture (SIFT, SuperPoint, and ORB) across four distinct pipeline conditions (Baseline, No CLAHE, No Symmetric Matching, and No Parallax Filter), we can directly quantify the reliance of each algorithm on specific pre-processing, filtering, and geometric failsafes.

##### **The Evaluation Scope: yunnan-garden-pavilion-P71**
This specific analysis evaluates the **yunnan-garden-pavilion-P71** dataset. As the most complex and unconstrained field evaluation in this study, this 71-image sequence captures the pavilion against a deep background of distant campus foliage and challenging outdoor illumination. It serves as the ultimate real-world stress test to evaluate how the removal of specific pipeline components degrades the system's ability to isolate rigid architectural structure from background noise and manage severe geometric drift over a prolonged trajectory.

#### **Setup & Data Extraction**

In [1]:
import os
import json
import pandas as pd
from pathlib import Path

In [2]:
EXPERIMENT_REPO_OWNER = "chuo-v"
EXPERIMENT_REPO_NAME = "incremental-sfm-experiments"

if 'google.colab' in str(get_ipython()):
    from google.colab import userdata

    if not os.path.exists(EXPERIMENT_REPO_NAME):
        try:
            token = userdata.get('GITHUB_TOKEN')
            repo_url = f"https://{token}@github.com/{EXPERIMENT_REPO_OWNER}/{EXPERIMENT_REPO_NAME}.git"
            print(f"Cloning private repository via PAT...")
        except userdata.SecretNotFoundError:
            repo_url = f"https://github.com/{EXPERIMENT_REPO_OWNER}/{EXPERIMENT_REPO_NAME}.git"
            print(f"Cloning public repository...")

        !git clone -q --depth 1 {repo_url} &> /dev/null

    os.chdir(EXPERIMENT_REPO_NAME)
    print(f"Environment Ready: Working directory set to {os.getcwd()}")
else:
    # Local Execution: Notebook is inside notebooks/03_comparative_analysis/
    # Move up two directories to reach the repository root
    repo_root = os.path.abspath("../../")
    os.chdir(repo_root)
    print(f"Environment Ready: Working directory set to {os.getcwd()}")

Cloning private repository via PAT...
Environment Ready: Working directory set to /content/incremental-sfm-experiments


In [3]:
repo_dir = Path.cwd()
dataset_name = "05_evaluation_yunnan_garden_pavilion_p71"
matchers = ["SIFT", "SuperPoint", "ORB"]
experiments = {
    "Baseline": repo_dir / "output" / "01_final" / dataset_name,
    "No CLAHE": repo_dir / "output" / "02_ablations" / "01_disable_clahe" / dataset_name,
    "No Symmetric Matching": repo_dir / "output" / "02_ablations" / "02_disable_symmetric_matching" / dataset_name,
    "No Parallax Filter": repo_dir / "output" / "02_ablations" / "03_disable_parallax_filter" / dataset_name
}

In [4]:
metrics_data = []

# Traverse the output directories and compile the JSON data
for exp_name, exp_path in experiments.items():
    for matcher in matchers:
        metrics_path = exp_path / "metrics" / f"run_metrics_{matcher}.json"

        if metrics_path.exists():
            with open(metrics_path, "r") as f:
                data = json.load(f)
                summary = data["results"]["summary"]

                metrics_data.append({
                    "Condition": exp_name,
                    "Matcher": matcher,
                    "Total 3D Points": summary.get("total_3d_points", 0),
                    "Overall RMSE (px)": round(summary.get("overall_rmse", 0.0), 3),
                    "Avg Frame Time (s)": round(summary.get("avg_frame_time_seconds", 0.0), 3),
                    "Mean Track Length": round(summary.get("mean_track_length", 0.0), 3),
                    "Tracking Inlier Ratio (%)": round(summary.get("tracking_inlier_ratio", 0.0) * 100, 1),
                    "Match Survival Rate (%)": round(summary.get("match_survival_rate", 0.0) * 100, 1)
                })
        else:
            print(f"Warning: Metrics not found for {matcher} under {exp_name} at {metrics_path}")

df = pd.DataFrame(metrics_data)

matcher_order = ["SIFT", "SuperPoint", "ORB"]
df['Matcher'] = pd.Categorical(df['Matcher'], categories=matcher_order, ordered=True)
condition_order = ["Baseline", "No CLAHE", "No Symmetric Matching", "No Parallax Filter"]
df['Condition'] = pd.Categorical(df['Condition'], categories=condition_order, ordered=True)
df = df.sort_values(['Matcher', 'Condition'])

display(df.style.hide(axis="index"))

Condition,Matcher,Total 3D Points,Overall RMSE (px),Avg Frame Time (s),Mean Track Length,Tracking Inlier Ratio (%),Match Survival Rate (%)
Baseline,SIFT,73521,3.987000,250.908000,2.423000,77.000000,15.200000
No CLAHE,SIFT,71099,4.917000,167.263000,2.404000,71.000000,17.200000
No Symmetric Matching,SIFT,85814,3.700000,115.013000,2.464000,75.300000,16.800000
No Parallax Filter,SIFT,70625,55.509000,126.201000,2.351000,75.900000,12.100000
Baseline,SuperPoint,68177,3.748000,64.892000,2.555000,54.900000,24.100000
No CLAHE,SuperPoint,66150,3.826000,53.288000,2.540000,53.800000,23.900000
No Symmetric Matching,SuperPoint,68177,3.748000,53.875000,2.555000,54.900000,24.100000
No Parallax Filter,SuperPoint,52719,17.866000,26.052000,2.398000,44.700000,13.300000
Baseline,ORB,36593,3.636000,111.469000,2.201000,80.400000,5.800000
No CLAHE,ORB,51553,2.977000,81.167000,2.211000,74.800000,6.300000


#### **Grouped Bar Chart Visualization**

In [5]:
%%capture
!pip install kaleido==0.2.1 plotly==5.24.1

In [6]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

In [7]:
colors = {
    "Baseline": "#1f77b4",               # Blue
    "No CLAHE": "#ff7f0e",               # Orange
    "No Symmetric Matching": "#d62728",  # Red
    "No Parallax Filter": "#2ca02c"      # Green
}

metrics_to_plot = [
    ("Total 3D Points", 1, 1),
    ("Overall RMSE (px)", 1, 2),
    ("Avg Frame Time (s)", 1, 3),
    ("Mean Track Length", 2, 1),
    ("Tracking Inlier Ratio (%)", 2, 2),
    ("Match Survival Rate (%)", 2, 3)
]

In [8]:
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=(
        "Total 3D Points",
        "Overall RMSE",
        "Avg Frame Time",
        "Mean Track Length",
        "Tracking Inlier Ratio %",
        "Match Survival Rate %"
    ),
    vertical_spacing=0.15
)

# Populate grouped bars per metric
for condition in condition_order:
    cond_df = df[df["Condition"] == condition]
    if cond_df.empty:
        continue

    for metric, row, col in metrics_to_plot:
        show_legend = True if (row == 1 and col == 1) else False

        fig.add_trace(
            go.Bar(
                x=cond_df["Matcher"],
                y=cond_df[metric],
                name=condition,
                marker_color=colors[condition],
                showlegend=show_legend
            ),
            row=row, col=col
        )

fig.update_layout(
    title_text=f"Ablation Comparison Synthesis: {dataset_name.split('_', 2)[-1]}",
    barmode='group',
    width=1400,
    height=800,
    template="plotly_white",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.05,
        xanchor="right",
        x=1
    )
)

fig.show()

# Export synthesis figure
vis_dir = Path("output/03_comparative_analysis") / dataset_name
vis_dir.mkdir(parents=True, exist_ok=True)
synthesis_chart_path = vis_dir / f"metrics_synthesis_chart_{dataset_name}.png"
fig.write_image(str(synthesis_chart_path), scale=2)
print(f"\nSynthesis chart exported to: {synthesis_chart_path}")


Synthesis chart exported to: output/03_comparative_analysis/05_evaluation_yunnan_garden_pavilion_p71/metrics_synthesis_chart_05_evaluation_yunnan_garden_pavilion_p71.png
